# Chapter 9: S3-like Object Storage
*System Design Interview Volume 2*

## TL;DR

S3-like object storage separates **metadata** (names, versions, ACL) from **object data** (binary blobs), inspired by the UNIX inode model. Objects are **immutable** and accessed via RESTful APIs. The data store appends small objects into large WAL-style files and uses a **placement service** with heartbeats to route writes to data nodes. **Erasure coding** (e.g., 8+4) achieves 11 nines of durability at 50% storage overhead versus 200% for triple replication. **Versioning** uses TIMEUUID rows with delete markers, and **multipart upload** enables resumable transfer of multi-GB files. A **garbage collector** compacts read-only files to reclaim space from deleted and orphaned data.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Bucket creation | Create globally unique named buckets |
| Functional | Object upload/download | PUT/GET objects via RESTful API |
| Functional | Object versioning | Keep multiple versions; delete markers |
| Functional | Object listing | List objects by prefix in a bucket |
| Non-functional | 100 PB capacity | Store 100 PB of data per year |
| Non-functional | 6 nines durability | 99.9999% data durability |
| Non-functional | 4 nines availability | 99.99% service availability |
| Non-functional | Storage efficiency | Minimize cost while maintaining reliability |

## Estimation: Storage and IOPS

In [ ]:
# Back-of-envelope: object count and metadata size
total_storage_pb = 100
storage_usage_ratio = 0.4
total_usable_mb = total_storage_pb * 1e9  # 100 PB in MB

# Object size distribution: 20% small (<1MB), 60% medium (1-64MB), 20% large (>64MB)
# Median sizes: 0.5 MB, 32 MB, 200 MB
weighted_avg_size_mb = 0.2 * 0.5 + 0.6 * 32 + 0.2 * 200
print(f"Weighted average object size: {weighted_avg_size_mb:.1f} MB")

effective_storage_mb = total_usable_mb * storage_usage_ratio
num_objects = effective_storage_mb / weighted_avg_size_mb
print(f"Estimated total objects: {num_objects / 1e9:.2f} billion")

# Metadata storage (1 KB per object)
metadata_tb = num_objects * 1e-3 / 1e6  # KB -> TB
print(f"Metadata storage needed: {metadata_tb:.2f} TB")

# Durability: 3-copy replication
annual_failure_rate = 0.0081
durability_3copy = 1 - annual_failure_rate ** 3
print(f"\n3-copy replication durability: {durability_3copy:.10f} ({(1 - durability_3copy):.2e} failure)")

# Erasure coding storage overhead comparison
replication_overhead = 2.0  # 200% extra (3 copies total)
erasure_overhead = 0.5       # 50% extra (4+2 scheme: 2 parities per 4 data)
print(f"\nReplication storage overhead: {replication_overhead*100:.0f}%")
print(f"Erasure coding (4+2) overhead: {erasure_overhead*100:.0f}%")
savings_pct = (replication_overhead - erasure_overhead) / replication_overhead * 100
print(f"Erasure coding saves: {savings_pct:.0f}% of redundancy storage")

## High-Level Design

```mermaid
graph TB
    Client([Client])
    LB[Load Balancer]
    API[API Service]
    IAM[IAM Service]

    subgraph MetadataStore["Metadata Store"]
        MDB[(Metadata DB)]
        MC[Metadata Cache]
    end

    subgraph DataStore["Data Store"]
        DR[Data Routing Service]
        PS[Placement Service]
        DN1[Data Node - Primary]
        DN2[Data Node - Secondary 1]
        DN3[Data Node - Secondary 2]
    end

    Client --> LB --> API
    API <--> IAM
    API <--> MDB
    API <--> MC
    API --> DR
    DR <--> PS
    DR --> DN1
    DN1 -->|replicate| DN2
    DN1 -->|replicate| DN3
    PS -.->|heartbeat| DN1
    PS -.->|heartbeat| DN2
    PS -.->|heartbeat| DN3
```

## Deep Dive

### Metadata and Data Separation

Object storage separates concerns like the UNIX inode model:
- **Metadata store** maps object name to UUID (like inode maps filename to block pointers)
- **Data store** persists binary data by UUID (like disk blocks)
- Each component scales and is optimized independently
- Metadata is mutable (versions, ACLs); data is immutable

### Data Node Internals

Small objects are **appended into large files** (WAL-style) instead of stored individually:

- Avoids wasting 4 KB disk blocks on sub-4KB objects
- Avoids inode exhaustion with millions of small files
- A **read-write file** accepts new objects until capacity threshold (several GBs), then becomes **read-only**
- Dedicated read-write files per CPU core avoid write serialization bottlenecks
- Each data node uses a local **SQLite** database to map UUID to (file_name, offset, size)

### Placement Service and Replication

```mermaid
graph TB
    subgraph VirtualCluster["Virtual Cluster Map"]
        ROOT[Root]
        DC1[Datacenter / AZ 1]
        DC2[Datacenter / AZ 2]
        DC3[Datacenter / AZ 3]
        R1[Rack 1]
        R2[Rack 2]
        R3[Rack 3]
        ROOT --> DC1 --> R1
        ROOT --> DC2 --> R2
        ROOT --> DC3 --> R3
    end

    PS[Placement Service<br/>5-7 nodes with Paxos/Raft]
    PS -->|heartbeats| R1
    PS -->|heartbeats| R2
    PS -->|heartbeats| R3
```

- Placement service runs as a 5-7 node cluster with Paxos/Raft consensus
- Virtual cluster map encodes physical topology (datacenter, rack, node)
- Replicas are placed in **different failure domains** (AZs) for maximum durability
- Nodes that miss heartbeats for 15 seconds are marked down

**Consistency-latency trade-off on replication:**

| Strategy | Consistency | Latency |
|---|---|---|
| ACK after all 3 nodes persist | Strongest | Highest (wait for slowest) |
| ACK after primary + 1 secondary | Medium | Medium |
| ACK after primary only | Eventual | Lowest |

### Erasure Coding

```mermaid
graph LR
    OBJ[Original Object]
    OBJ -->|split| D1[d1]
    OBJ -->|split| D2[d2]
    OBJ -->|split| D3[d3]
    OBJ -->|split| D4[d4]
    D1 --> CALC[Parity Calc]
    D2 --> CALC
    D3 --> CALC
    D4 --> CALC
    CALC --> P1[p1]
    CALC --> P2[p2]

    D1 --> FD1[Failure Domain 1]
    D2 --> FD2[Failure Domain 2]
    D3 --> FD3[Failure Domain 3]
    D4 --> FD4[Failure Domain 4]
    P1 --> FD5[Failure Domain 5]
    P2 --> FD6[Failure Domain 6]
```

**(4+2) erasure coding**: data is split into 4 chunks, 2 parity chunks computed via Reed-Solomon. Any 4 of 6 chunks can reconstruct the full object. With (8+4) coding, 11 nines of durability at 50% overhead.

| Property | 3-Copy Replication | Erasure Coding (8+4) |
|---|---|---|
| Durability | ~6 nines | ~11 nines |
| Storage overhead | 200% | 50% |
| Write latency | Lower (no parity calc) | Higher (parity computation) |
| Read latency | Lower (single node read) | Higher (multi-node gather) |
| Complexity | Simple | Complex |

### Object Versioning

When versioning is enabled, each PUT creates a **new metadata row** (same bucket + name, new UUID + TIMEUUID). DELETE inserts a **delete marker** as the latest version. GET on a delete marker returns 404. All previous versions remain and old object data is never immediately removed.

### Multipart Upload

Large files (multi-GB) are split into parts uploaded independently:

1. Client calls initiate -- server returns **uploadID**
2. Each part is uploaded with the uploadID -- server returns **ETag** (MD5)
3. Client sends completion request with part numbers and ETags
4. Server reassembles the object from parts
5. Garbage collector cleans orphaned/abandoned parts

### Metadata Sharding

- **Bucket table**: small (around 10 GB for 1M users x 10 buckets); single DB with read replicas
- **Object table**: too large for one DB; shard by **hash(bucket_name, object_name)**
- Listing objects by prefix requires querying all shards and aggregating
- Optimization: denormalize listing data into a **separate table sharded by bucket_id**

### Garbage Collection and Compaction

```mermaid
graph LR
    subgraph Before["Before Compaction"]
        F1["data/b<br/>contains deleted + live objects"]
    end

    subgraph After["After Compaction"]
        F2["data/d<br/>only live objects copied"]
    end

    GC[Garbage Collector]
    F1 --> GC -->|copy live objects| F2
    GC -->|update mapping table| DB[(object_mapping)]
```

Garbage targets: lazy-deleted objects, orphaned multipart parts, corrupted data. Compaction copies live objects from many small read-only files into fewer large files, then atomically updates the mapping table.

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Durability method | 3-copy replication: simple, fast | Erasure coding: higher durability, lower cost, complex |
| Replication ACK | ACK all: strong consistency | ACK primary: low latency, eventual consistency |
| Small object storage | One file per object: simple | WAL-style append: efficient but complex lookup |
| Metadata sharding | By bucket_id: hot spots | By hash(bucket, object): even distribution, hard listing |
| Large uploads | Single PUT: simple | Multipart: resumable, parallel, needs GC |
| Versioning | Disabled: less storage | Enabled: safe rollback, more metadata and storage |

## Key Takeaways

1. **Separation of metadata and data** is the foundational design principle -- each scales independently
2. **Immutability** simplifies replication, caching, and consistency reasoning
3. **Erasure coding** delivers dramatically better durability per byte of storage than replication
4. **WAL-style file append** solves small-file inode exhaustion and disk-block waste
5. **Versioning via TIMEUUID** is append-only in the metadata store -- no destructive updates
6. **Multipart upload** makes large object ingestion resumable and parallelizable
7. **Garbage collection** is essential when deletes are lazy and uploads can be abandoned

## Related Concepts

- [[object-storage-fundamentals]] -- core concepts: buckets, objects, immutability
- [[metadata-data-separation]] -- UNIX inode-inspired architecture
- [[data-persistence-and-routing]] -- placement service, data nodes, WAL append
- [[erasure-coding]] -- Reed-Solomon durability at lower storage cost
- [[object-versioning]] -- TIMEUUID versioning with delete markers
- [[multipart-upload]] -- resumable upload for large files
- [[garbage-collection-compaction]] -- compaction and lazy deletion